# Run Trajectory Statistic Probe

该 notebook 用于 Colab GPU 环境中的阶段 3 formal replay 验证。它只调用仓库 CLI / helper, 不在 notebook 内直接写正式 records、thresholds、tables 或 reports。


## 00 Runtime Identity And User Config


In [ ]:
from pathlib import Path
import os

NOTEBOOK_MANUAL_CONFIG = {
    'repo_url': 'https://github.com/RICHAAARC/TSTW-VW.git',
    'repo_branch': 'main',
    'repo_root': '/content/TSTW-VW',
    'drive_root': '/content/drive/MyDrive/TSTW',
    'stage2_frozen_package_path': '',
    'stage2_result_root': '/content/drive/MyDrive/TSTW/results/real_video_vae_latent_probe',
    'runtime_profile': 'formal',
    'samples_per_role': 2,
    'run_id_template': 'trajectory_statistic_probe_formal_gpu_validation_utc_time_short_commit',
}

REPO_URL = os.environ.get('TSTW_REPO_URL', NOTEBOOK_MANUAL_CONFIG['repo_url'])
REPO_BRANCH = os.environ.get('TSTW_REPO_BRANCH', NOTEBOOK_MANUAL_CONFIG['repo_branch'])
REPO_ROOT = Path(os.environ.get('TSTW_REPO_ROOT', NOTEBOOK_MANUAL_CONFIG['repo_root']))
DRIVE_ROOT = Path(os.environ.get('TSTW_DRIVE_ROOT', NOTEBOOK_MANUAL_CONFIG['drive_root']))
STAGE2_FROZEN_PACKAGE_PATH_TEXT = os.environ.get('TSTW_STAGE2_FROZEN_PACKAGE_PATH', NOTEBOOK_MANUAL_CONFIG['stage2_frozen_package_path']).strip()
STAGE2_RESULT_ROOT = Path(os.environ.get('TSTW_STAGE2_RESULT_ROOT', NOTEBOOK_MANUAL_CONFIG['stage2_result_root']))
STAGE2_FROZEN_PACKAGE_PATH = Path(STAGE2_FROZEN_PACKAGE_PATH_TEXT) if STAGE2_FROZEN_PACKAGE_PATH_TEXT else None
RUNTIME_PROFILE = os.environ.get('TSTW_TRAJECTORY_RUNTIME_PROFILE', NOTEBOOK_MANUAL_CONFIG['runtime_profile'])
SAMPLES_PER_ROLE = int(os.environ.get('TSTW_TRAJECTORY_SAMPLES_PER_ROLE', str(NOTEBOOK_MANUAL_CONFIG['samples_per_role'])))
RUN_ID_TEMPLATE = os.environ.get('TSTW_TRAJECTORY_RUN_ID_TEMPLATE', NOTEBOOK_MANUAL_CONFIG['run_id_template'])
RUN_ID_OVERRIDE = os.environ.get('TSTW_TRAJECTORY_RUN_ID')
LOCAL_INPUT_ROOT = REPO_ROOT / 'outputs' / 'runtime_inputs' / 'stage2_frozen_baseline_for_trajectory_probe'

print('REPO_ROOT =', REPO_ROOT)
print('STAGE2_RESULT_ROOT =', STAGE2_RESULT_ROOT)
print('STAGE2_FROZEN_PACKAGE_PATH =', STAGE2_FROZEN_PACKAGE_PATH)
print('RUNTIME_PROFILE =', RUNTIME_PROFILE)
print('RUN_ID_TEMPLATE =', RUN_ID_TEMPLATE)
print('RUN_ID_OVERRIDE =', RUN_ID_OVERRIDE)


## 01 Mount Google Drive


In [ ]:
from google.colab import drive

drive.mount('/content/drive')
assert DRIVE_ROOT.exists(), f'Drive root not found: {DRIVE_ROOT}'


## 02 Clone Or Update Repository


In [ ]:
import subprocess
from datetime import datetime, timezone

if REPO_ROOT.exists():
    subprocess.run(['git', 'fetch', '--all'], cwd=REPO_ROOT, check=True)
    subprocess.run(['git', 'checkout', REPO_BRANCH], cwd=REPO_ROOT, check=True)
    subprocess.run(['git', 'pull', '--ff-only'], cwd=REPO_ROOT, check=True)
else:
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, REPO_URL, str(REPO_ROOT)], check=True)

GIT_COMMIT = subprocess.check_output(
    ['git', 'rev-parse', '--short=7', 'HEAD'],
    cwd=REPO_ROOT,
    text=True,
).strip()
UTC_TIME = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RUN_ID = RUN_ID_OVERRIDE or RUN_ID_TEMPLATE.replace('utc_time', UTC_TIME).replace('short_commit', GIT_COMMIT)
OUTPUT_ROOT = REPO_ROOT / 'outputs' / 'runs' / RUN_ID
PACKAGE_ROOT = DRIVE_ROOT / 'results' / 'trajectory_statistic_probe' / RUN_ID / 'packages'

print('GIT_COMMIT =', GIT_COMMIT)
print('UTC_TIME =', UTC_TIME)
print('RUN_ID =', RUN_ID)
print('OUTPUT_ROOT =', OUTPUT_ROOT)
print('PACKAGE_ROOT =', PACKAGE_ROOT)


## 03 Install Runtime Dependencies


In [ ]:
import subprocess

subprocess.run(['python', '-m', 'pip', 'install', '-q', 'pytest'], check=True)


## 04 Verify Repository Contract


In [ ]:
import os
import subprocess

repo_env = dict(os.environ)
repo_env['PYTHONPATH'] = str(REPO_ROOT) + os.pathsep + repo_env.get('PYTHONPATH', '')
repo_env['PYTHONUTF8'] = '1'
repo_env['PYTHONIOENCODING'] = 'utf-8'
subprocess.run(['python', 'tools/harness/validate_project_contract.py'], cwd=REPO_ROOT, env=repo_env, check=True)


## 05 Run Trajectory Smoke Tests


In [ ]:
subprocess.run([
    'python', '-m', 'pytest', '-q',
    'tests/functional/test_trajectory_mechanism_audit.py',
    'tests/functional/test_trajectory_source_access_guard.py',
    'tests/integration/test_trajectory_formal_replay_smoke.py',
    '-m', 'quick or integration',
], cwd=REPO_ROOT, env=repo_env, check=True)


## 06 Extract Stage Two Frozen Baseline


In [ ]:
import sys

sys.path.insert(0, str(REPO_ROOT))
from paper_workflow.notebook_utils import trajectory_statistic_probe_workflow as trajectory_workflow

if STAGE2_FROZEN_PACKAGE_PATH is None:
    stage2_package_candidates = sorted(
        STAGE2_RESULT_ROOT.glob('*/packages/real_video_vae_latent_probe_formal.zip'),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )
    if not stage2_package_candidates:
        raise FileNotFoundError(
            'No stage 2 frozen zip package found under '
            f'{STAGE2_RESULT_ROOT}. Set TSTW_STAGE2_FROZEN_PACKAGE_PATH explicitly.'
        )
    STAGE2_FROZEN_PACKAGE_PATH = stage2_package_candidates[0]

frozen_baseline_root = trajectory_workflow.extract_frozen_baseline_package(
    STAGE2_FROZEN_PACKAGE_PATH,
    LOCAL_INPUT_ROOT,
)
print('STAGE2_FROZEN_PACKAGE_PATH =', STAGE2_FROZEN_PACKAGE_PATH)
print('frozen_baseline_root =', frozen_baseline_root)


## 07 Run Trajectory Formal Replay


In [ ]:
decision = trajectory_workflow.run_formal_replay_cli(
    repository_root=REPO_ROOT,
    frozen_baseline_root=frozen_baseline_root,
    output_root=OUTPUT_ROOT,
    runtime_profile=RUNTIME_PROFILE,
    samples_per_role=SAMPLES_PER_ROLE,
)
print(decision)


## 08 Package Trajectory Results


In [ ]:
archive_path = trajectory_workflow.package_trajectory_probe_run(
    run_root=OUTPUT_ROOT,
    package_root=PACKAGE_ROOT,
    package_name=RUN_ID,
)
print('archive_path =', archive_path)


## 09 Print Final Summary


In [ ]:
print('Stage3ImplementationDecision =', decision.get('Stage3ImplementationDecision'))
print('Stage3MechanismDecision =', decision.get('Stage3MechanismDecision'))
print('Stage2DependencyStatus =', decision.get('Stage2DependencyStatus'))
print('trajectory_source_kind =', decision.get('trajectory_source_kind'))
print('formal_trajectory_source_status =', decision.get('formal_trajectory_source_status'))
print('TrajectoryMechanismGateSummary =', decision.get('TrajectoryMechanismGateSummary'))
print('Stage3MechanismBlockingReasons =', decision.get('Stage3MechanismBlockingReasons'))
print('NextAllowedStageByTrajectory =', decision.get('NextAllowedStageByTrajectory'))
print('Download package:', archive_path)
